In [8]:
import os
print("root listing:", sorted(os.listdir('.'))[:20])
# 检查关键输出目录
print("data/sensor/demo_output:", os.path.exists('data/sensor/demo_output'), os.listdir('data/sensor/demo_output')[:20] if os.path.exists('data/sensor/demo_output') else 'no dir')
print("data/icf/demo_output:", os.path.exists('data/icf/demo_output'), os.listdir('data/icf/demo_output') if os.path.exists('data/icf/demo_output') else 'no dir')
print("data/fusion/demo_output:", os.path.exists('data/fusion/demo_output'), os.listdir('data/fusion/demo_output') if os.path.exists('data/fusion/demo_output') else 'no dir')

root listing: ['.git', '.gitignore', '.idea', '.ipynb_checkpoints', '.venv', '.venv310', 'README.md', 'check_outputs.ipynb', 'code', 'data', 'docs', 'fusion', 'main.py', 'requirements.txt', 'run_fusion_demo.py', 'test.py', 'venv']
data/sensor/demo_output: True ['imu_action_scores.csv']
data/icf/demo_output: True ['icf_prediction_results.png', 'icf_time_series.csv']
data/fusion/demo_output: True ['advanced_metrics.csv', 'performance_comparison.png', 'test_predictions.csv', 'threshold_evolution.png', 'threshold_trace.csv', 'training_dynamics.png', 'weight_evolution.png', 'weight_trace.csv']


In [1]:
import pandas as pd
imu_path = 'data/sensor/demo_output/imu_action_scores.csv'   # 或 data/imu/...
df_imu = pd.read_csv(imu_path)
print("IMU columns:", df_imu.columns.tolist())
print(df_imu.head())
print(df_imu.describe(include='all').loc[['count','unique']] if 'action' in df_imu.columns else df_imu.describe())

IMU columns: ['patient_id', 'action_type', 'quality_score']
  patient_id action_type  quality_score
0      P0001           站         0.9996
1      P0002           站         0.7636
2      P0003           站         0.9970
3      P0004           站         0.9995
4      P0005           站         0.9996
       quality_score
count    2947.000000
mean        0.973917
std         0.074878
min         0.432100
25%         0.992650
50%         0.998800
75%         0.999700
max         1.000000


In [20]:
import pandas as pd

path = "data/icf/demo_output/icf_time_series.csv"
df = pd.read_csv(path)

start_date = pd.to_datetime("2023-01-01")

dates = []

for pid, group in df.groupby("patient_id"):
    
    group = group.sort_values("time_step")
    
    for i in range(len(group)):
        
        date = start_date + pd.Timedelta(days=i*7)
        
        dates.append(date)

df["assessment_date"] = dates

df.to_csv(path,index=False)

print("assessment_date added")
df.head()

assessment_date added


,patient_id,time_step,rehab_phase,icf_total,rom,vas,assessment_date
0,P001,0,早期,56,22.3,6,2023-01-01
1,P001,1,早期,71,33.2,5,2023-01-08
2,P001,2,中期,111,74.3,8,2023-01-15
3,P001,3,中期,119,92.8,5,2023-01-22
4,P001,4,中期,137,111.1,4,2023-01-29


In [24]:
icf_path = 'data/icf/demo_output/icf_time_series.csv'
df_icf = pd.read_csv(icf_path)
print("ICF cols:", df_icf.columns.tolist())
print("unique patients:", df_icf['patient_id'].nunique())
print("per-patient counts min/median/max:", df_icf.groupby('patient_id').size().agg(['min','median','max']))
# 检查日期列（若有）
df_icf['assessment_date'] = pd.to_datetime(df_icf['assessment_date'], errors='coerce')
print("null dates:", df_icf['assessment_date'].isna().sum())

ICF cols: ['patient_id', 'time_step', 'rehab_phase', 'icf_total', 'rom', 'vas', 'assessment_date']
unique patients: 50
per-patient counts min/median/max: min       6.0
median    6.0
max       6.0
dtype: float64
null dates: 0


In [4]:
fusion_pred = 'data/fusion/demo_output/test_predictions.csv'
fusion_metrics = 'data/fusion/demo_output/advanced_metrics.csv'
if os.path.exists(fusion_pred):
    df_fp = pd.read_csv(fusion_pred)
    print("fusion pred cols:", df_fp.columns.tolist())
    print(df_fp.head())
if os.path.exists(fusion_metrics):
    print(pd.read_csv(fusion_metrics).T)

fusion pred cols: ['y_true', 'y_prob_advanced', 'y_pred_advanced', 'y_prob_static', 'y_pred_static']
   y_true  y_prob_advanced  y_pred_advanced  y_prob_static  y_pred_static
0       0         0.466260                0       0.488835              0
1       0         0.522378                1       0.575717              1
2       0         0.430943                0       0.438377              0
3       0         0.472274                1       0.506319              1
4       0         0.389193                0       0.426481              0
                    0                1
model     static_rule  advanced_bandit
accuracy     0.777083          0.83125
f1           0.792233         0.827292
auc          0.965055         0.964967


In [19]:
import pandas as pd

imu = pd.read_csv("data/sensor/demo_output/imu_action_scores.csv")
icf = pd.read_csv("data/icf/demo_output/icf_time_series.csv")

print("IMU patients:", imu["patient_id"].nunique())
print("ICF patients:", icf["patient_id"].nunique())

IMU patients: 2947
ICF patients: 50


In [25]:
import pandas as pd
imu = pd.read_csv('data/sensor/demo_output/imu_action_scores.csv')
icf = pd.read_csv('data/icf/demo_output/icf_time_series.csv')

print("IMU rows:", len(imu))
print("IMU unique patient_id:", imu['patient_id'].nunique())
print("ICF unique patient_id:", icf['patient_id'].nunique())

# 看两边 patient_id 的交集
imu_ids = set(imu['patient_id'].unique())
icf_ids = set(icf['patient_id'].unique())
print("intersection count:", len(imu_ids & icf_ids))
print("example intersection (<=20):", list(imu_ids & icf_ids)[:20])

IMU rows: 2947
IMU unique patient_id: 2947
ICF unique patient_id: 50
intersection count: 0
example intersection (<=20): []
